<a href="https://colab.research.google.com/github/AidaAm005/NHANES-Tobacco-Biomarkers/blob/phase-1/notebooks/phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import sys
from google.colab import drive

# ۱. اتصال به گوگل درایو (هنگام اجرا یک پنجره تایید دسترسی باز می‌شود)
drive.mount('/content/drive')

# ۲. تعریف مسیر پوشه اصلی پروژه در گوگل درایو
PROJECT_ROOT = '/content/drive/MyDrive/NHANES-Tobacco-Biomarkers'

# ۳. تغییر دایرکتوری کاری کولب به ریشه پروژه
os.chdir(PROJECT_ROOT)

# ۴. افزودن ریشه پروژه به مسیرهای پایتون برای شناسایی پکیج src
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("مسیر کاری فعلی:", os.getcwd())
print("فایل‌های موجود در پوشه src:", os.listdir('src'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
مسیر کاری فعلی: /content/drive/MyDrive/NHANES-Tobacco-Biomarkers
فایل‌های موجود در پوشه src: ['data_cleaner.py', 'data_loader.py']


In [4]:
import pandas as pd
from src.data_loader import fetch_raw_datasets, merge_datasets

# ۱. دریافت فایل‌ها از سرور CDC و گزارش ابعاد اولیه پیش از ادغام
raw_datasets, df_pre_merge = fetch_raw_datasets()
print("=== جدول الف-۱ (بخش اول): مشخصات و تعداد افراد هر فایل پیش از ادغام ===")
display(df_pre_merge)

# ۲. ادغام فایل‌ها بر پایه DEMO_L با کلید مشترک SEQN
df_merged_raw, df_merge_report = merge_datasets(raw_datasets)
print("\n=== جدول الف-۱ (بخش دوم): گزارش تعداد افراد قبل و بعد از هر مرحله ادغام ===")
display(df_merge_report)

=== جدول الف-۱ (بخش اول): مشخصات و تعداد افراد هر فایل پیش از ادغام ===


,File_Name,Total_Rows_Raw,Unique_SEQN_Raw,Total_Columns_Raw,Selected_Columns_Count
0,DEMO_L,11933,11933,27,10
1,SMQ_L,9015,9015,9,4
2,SMQRTU_L,7199,7199,21,9
3,SMQFAM_L,11933,11933,3,2
4,COT_L,8493,8493,5,5
5,PBCD_L,8727,8727,17,8
6,CBC_L,8727,8727,23,3



=== جدول الف-۱ (بخش دوم): گزارش تعداد افراد قبل و بعد از هر مرحله ادغام ===


,Merge_Step,Merged_File,Merge_Key,Total_Rows_Left_Join,Matched_Participants_in_Step,Cumulative_Complete_Overlap (Inner)
0,1. Base File: DEMO_L,DEMO_L,SEQN,11933,11933,11933
1,2. Merge SMQ_L,SMQ_L,SEQN,11933,9015,9015
2,3. Merge SMQRTU_L,SMQRTU_L,SEQN,11933,7199,7199
3,4. Merge SMQFAM_L,SMQFAM_L,SEQN,11933,11933,7199
4,5. Merge COT_L,COT_L,SEQN,11933,8493,7199
5,6. Merge PBCD_L,PBCD_L,SEQN,11933,8727,7199
6,7. Merge CBC_L,CBC_L,SEQN,11933,8727,7199


In [5]:
from src.data_cleaner import (
    fix_sas_zeros,
    audit_and_clean_special_codes,
    audit_duplicates_and_logic,
    audit_missing_and_ranges,
    audit_lod_censored_values
)

# ۱. اصلاح مقادیر علمی بسیار کوچک (صفرهای فرمت SAS XPT)
df_step1, n_tiny_zeros = fix_sas_zeros(df_merged_raw)
print(f"تعداد مقادیر ممیز شناور نزدیک به صفر که به 0.0 دقیق تبدیل شدند: {n_tiny_zeros} مورد\n")

# ۲. شناسایی کدهای ویژه «امتناع (7/77/777)» و «نامعلوم (9/99/999)» و تبدیل به NaN
df_cleaned, df_special_codes = audit_and_clean_special_codes(df_step1)
print("=== جدول ۵-۱: ممیزی کدهای ویژه نامعلوم و پاسخ نداده در پرسش‌نامه‌ها ===")
display(df_special_codes)

# ۳. بررسی رکوردهای تکراری، مقادیر غیرممکن و ناسازگاری‌های منطقی
df_logic_audit = audit_duplicates_and_logic(df_cleaned)
print("\n=== جدول ۵-۲: ممیزی رکوردهای تکراری، مقادیر غیرممکن و سازگاری منطقی ===")
display(df_logic_audit)

# ۴. ممیزی داده‌های گمشده و بازه مقادیر (در کل نمونه و در بزرگسالان ۲۰ سال به بالا)
df_missing_all = audit_missing_and_ranges(df_cleaned, subset_adults=False)
df_missing_adults = audit_missing_and_ranges(df_cleaned, subset_adults=True)
print("\n=== جدول ۵-۳ (الف): ممیزی داده‌های گمشده و دامنه مقادیر در کل نمونه ادغام‌شده ===")
display(df_missing_all)
print("\n=== جدول ۵-۳ (ب): ممیزی داده‌های گمشده و دامنه مقادیر در بزرگسالان (۲۰ سال و بالاتر) ===")
display(df_missing_adults)

# ۵. ممیزی مقادیر سانسورشده زیر حد تشخیص آزمایشگاه (Below LOD)
df_lod_audit = audit_lod_censored_values(df_cleaned, subset_adults=True)
print("\n=== جدول ۵-۴: ممیزی مقادیر زیر حد تشخیص دستگاه آزمایشگاه (Below LOD) ===")
display(df_lod_audit)

تعداد مقادیر ممیز شناور نزدیک به صفر که به 0.0 دقیق تبدیل شدند: 37173 مورد

=== جدول ۵-۱: ممیزی کدهای ویژه نامعلوم و پاسخ نداده در پرسش‌نامه‌ها ===


,Variable,Special_Codes_Checked,Refused_Count (7/77/777),Dont_Know_Count (9/99/999),Total_Converted_to_NaN
0,DMDEDUC2,"[7, 9]",0,11,11
1,SMQ020,"[7, 9]",7,7,14
2,SMQ040,"[7, 9]",0,0,0
3,SMD650,"[777, 999]",0,9,9
4,SMQ681,"[7, 9]",1,2,3
5,SMQ710,"[7, 9, 77, 99]",1,1,2
6,SMQ720,"[777, 999]",1,1,2
7,SMQ845,"[7, 9]",0,1,1
8,SMQ846,"[7, 9]",1,4,5
9,SMQ849,"[7, 9, 77, 99]",0,1,1



=== جدول ۵-۲: ممیزی رکوردهای تکراری، مقادیر غیرممکن و سازگاری منطقی ===


,Check_Type,Count,Decision
0,Duplicate Full Rows,0,No action needed (0 duplicates)
1,Duplicate SEQN IDs,0,1-to-1 merge verified
2,Impossible Negative Values,0,All quantitative values within valid physical ...
3,Questionnaire Skip-Pattern Violation (SMQ020=2...,0,Structural missingness confirmed
4,Discordant Self-Report (Non-smoker in SMQ but ...,188,Classify based on active recent use (SMQ681=1)...
5,Impossible Recent Days (>5 days in last 5 days),0,"Valid range [0, 5] confirmed"



=== جدول ۵-۳ (الف): ممیزی داده‌های گمشده و دامنه مقادیر در کل نمونه ادغام‌شده ===


,Variable,Total_N,Valid_N,Missing_Count,Missing_Percent (%),Min_Value,Max_Value
0,RIDAGEYR,11933,11933,0,0.00,0.0000,80.0000
1,RIAGENDR,11933,11933,0,0.00,1.0000,2.0000
2,RIDRETH3,11933,11933,0,0.00,1.0000,7.0000
3,DMDEDUC2,11933,7783,4150,34.78,1.0000,5.0000
4,INDFMPIR,11933,9892,2041,17.10,0.0000,5.0000
5,WTINT2YR,11933,11933,0,0.00,4584.4632,170968.3432
6,WTMEC2YR,11933,11933,0,0.00,0.0000,227108.2970
7,SDMVPSU,11933,11933,0,0.00,1.0000,2.0000
8,SDMVSTRA,11933,11933,0,0.00,173.0000,187.0000
9,SMQ020,11933,8121,3812,31.95,1.0000,2.0000



=== جدول ۵-۳ (ب): ممیزی داده‌های گمشده و دامنه مقادیر در بزرگسالان (۲۰ سال و بالاتر) ===


,Variable,Total_N,Valid_N,Missing_Count,Missing_Percent (%),Min_Value,Max_Value
0,RIDAGEYR,7809,7809,0,0.00,20.0000,80.0000
1,RIAGENDR,7809,7809,0,0.00,1.0000,2.0000
2,RIDRETH3,7809,7809,0,0.00,1.0000,7.0000
3,DMDEDUC2,7809,7783,26,0.33,1.0000,5.0000
4,INDFMPIR,7809,6489,1320,16.90,0.0000,5.0000
5,WTINT2YR,7809,7809,0,0.00,4584.4632,170968.3432
6,WTMEC2YR,7809,7809,0,0.00,0.0000,227108.2970
7,SDMVPSU,7809,7809,0,0.00,1.0000,2.0000
8,SDMVSTRA,7809,7809,0,0.00,173.0000,187.0000
9,SMQ020,7809,7780,29,0.37,1.0000,2.0000



=== جدول ۵-۴: ممیزی مقادیر زیر حد تشخیص دستگاه آزمایشگاه (Below LOD) ===


,Biomarker,Description,Comment_Code_Var,Total_Tested,Below_LOD_Count (Code=1),Below_LOD_Percent (%),CDC_Imputed_Value (LOD/sqrt(2))
0,LBXCOT,Serum Cotinine (ng/mL),LBDCOTLC,5521,2180,39.49,0.007
1,LBXHCOT,Serum Hydroxycotinine (ng/mL),LBDHCOLC,5522,2177,39.42,0.003
2,LBXBCD,Blood Cadmium (ug/L),LBDBCDLC,5770,45,0.78,0.046
3,LBXBPB,Blood Lead (ug/dL),LBDBPBLC,5770,0,0.00,NaN
4,LBXBSE,Blood Selenium (ug/L),LBDBSELC,5770,0,0.00,NaN
